# 05 — Cancellation, Products of Experts, and Verification Scaling
## Defect Cancellation Caveats, PoE vs MoE Trade-Offs, and Optimal Verification Density

**Series:** `001_finite_obstruction_calculus` / `012_lggr_i_evaluated_path_kernels`  
**Author:** Jeremy H. Carroll  
**Status:** Living Educational Supplement  
**Anchor Tag:** `sealed-derivation-v1` (`3100c4e`, cut 2026-08-17)

---

> ### **Claim-Typing Legend**
> - **[Pinned]**: Verified directly by executable Python functions in `code/epk_core.py` and passing test suites.
> - **[Proved]**: Mechanized in Lean 4 under `PaperAE004` (reported kernel axioms: `propext`, `Classical.choice`, `Quot.sound` or axiom-free).
> - **[Declared]**: An operational definition, architectural parameter, or illustrative teaching model chosen by the author.
> - **[Open]**: A mathematical or empirical question not discharged in this package.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. The Aggregate Cancellation Caveat (§4)

In Section 4 of the paper, we prove a vital mathematical asymmetry:

$$\Delta_E(\gamma) \equiv 0 \implies R_{\mathrm{Eval}} = 0 \qquad \text{BUT} \qquad R_{\mathrm{Eval}} = 0 \not\implies \Delta_E(\gamma) \equiv 0$$

- If evaluators factorize pointwise everywhere ($\Delta_E = 0$ for all paths), then the Evaluator Residual $R_{\mathrm{Eval}}$ is guaranteed to be zero.
- **However, if $R_{\mathrm{Eval}} = 0$, you CANNOT assume evaluators factorize!**

### Why? Defect Cancellation
Recall the formula for the action residual:
$$R_{\mathrm{Eval}} = \sum_{\gamma} \Delta_E(\gamma) \cdot e^{-\beta \mathcal{A}[\gamma]}$$

Because $\Delta_E \in \{-1, 0, +1\}$ is **signed**, a path with a Positive Gluing Defect ($\Delta_E = +1$) can mathematically cancel out a path with a Negative Gluing Defect ($\Delta_E = -1$) if their weighted costs match!

Let's build this exact counterexample in code:


In [ ]:
# Set up 2 paths with identical actions (A = 1.0), but opposite defects:
s_a, s_b1, s_b2, s_c = map(State, ["Sa", "Sb1", "Sb2", "Sc"])

# Path 1: Sa -> Sb1 -> Sc (Action: 0.5 + 0.5 = 1.0)
p1_left = TrajectorySegment(s_a, s_b1, ("step1_a",), raw_action=0.5)
p1_right = TrajectorySegment(s_b1, s_c, ("step2_a",), raw_action=0.5)
comp1 = ComposedTrajectory(p1_left, p1_right)

# Path 2: Sa -> Sb2 -> Sc (Action: 0.5 + 0.5 = 1.0)
p2_left = TrajectorySegment(s_a, s_b2, ("step1_b",), raw_action=0.5)
p2_right = TrajectorySegment(s_b2, s_c, ("step2_b",), raw_action=0.5)
comp2 = ComposedTrajectory(p2_left, p2_right)

# Evaluator setup designed to create exact cancellation:
# Path 1 has PGD: Local fails (E1=0), Global passes (Eg=1) -> Delta_E = +1
# Path 2 has NGD: Local passes (E1=1, E2=1), Global fails (Eg=0) -> Delta_E = -1
eval_set_cancelling = EvaluatorSet(
    eval_1=lambda seg: 0 if seg == p1_left else 1,
    eval_2=lambda seg: 1,
    eval_g=lambda comp: 1 if comp == comp1 else 0
)

# Compute Evaluator Residual
r_eval = EvaluatedPathKernel.compute_action_evaluator_residual([comp1, comp2], eval_set_cancelling, beta=1.0)

print(f"Path 1 (PGD) Defect contribution: +1 * exp(-1.0) = +{np.exp(-1.0):.6f}")
print(f"Path 2 (NGD) Defect contribution: -1 * exp(-1.0) = -{np.exp(-1.0):.6f}")
print(f"Net Aggregate Residual R_Eval    : {r_eval:.1e}")

assert abs(r_eval) < 1e-15
print("\n[CAUTION CONFIRMED] R_Eval is EXACTLY ZERO, yet 100% of paths have evaluator defects!")


## 2. Verification Architecture: Product-of-Experts vs Mixture-of-Experts (§6.3)

When designing multi-agent verification pipelines, how should we combine multiple verifiers or reviewers?

### Product of Experts (PoE) — The High-Security Gate
In a Product of Experts architecture, all $K$ verifiers must agree to accept a solution:
$$E_{\mathrm{PoE}}(\gamma) = \prod_{e=1}^K E_e(\gamma)$$

For probabilistic candidate distributions:
$$p_{\mathrm{PoE}}(W) = \frac{\prod_{e=1}^K p_e(W)}{\sum_\eta \prod_{e=1}^K p_e(\eta)}$$

- **Strength:** Unforgiving precision. If even one expert spots a critical bug ($p_e = 0$), the entire candidate is suppressed.
- **Failure Mode:** **Refusal Accumulation**. If each reviewer has an independent false-rejection rate of $5\%$, a pipeline with $K=5$ reviewers will reject good solutions with probability $1 - (0.95)^5 \approx 22.6\%$!

### Mixture of Experts (MoE) — Robust Consensus
In a Mixture of Experts architecture, verdicts or scores are combined via weighted routing or threshold voting:
$$E_{\mathrm{MoE}}(\gamma) = \sum_{e=1}^K w_e E_e(\gamma)$$

- **Strength:** High throughput, resilient to individual noisy reviewers.
- **Failure Mode:** Stealth minority failure. If 2 reviewers miss a security flaw that 1 reviewer caught, majority voting accepts the flawed code.


In [ ]:
# Candidate Ensemble: 3 candidate solutions evaluated by 3 expert judges
# Candidate 0: Solid consensus [0.8, 0.7, 0.9]
# Candidate 1: One expert rejects [0.9, 0.0, 0.8]  (Hard refusal!)
# Candidate 2: Mediocre approval [0.4, 0.5, 0.4]

candidates = [
    [0.8, 0.7, 0.9],
    [0.9, 0.0, 0.8],
    [0.4, 0.5, 0.4]
]

poe_dist = ReliabilityModels.normalized_product_of_experts(candidates)

df_poe = pd.DataFrame({
    "Candidate": ["Candidate 0 (Consensus)", "Candidate 1 (Single Flaw)", "Candidate 2 (Mediocre)"],
    "Expert 1": [0.8, 0.9, 0.4],
    "Expert 2": [0.7, 0.0, 0.5],
    "Expert 3": [0.9, 0.8, 0.4],
    "Raw PoE Product": [0.8*0.7*0.9, 0.9*0.0*0.8, 0.4*0.5*0.4],
    "Normalized PoE Share": [f"{p:.1%}" for p in poe_dist]
})

print("=== NORMALIZED PRODUCT-OF-EXPERTS (PoE) DISTRIBUTION ===")
df_poe
# Assert expected normalized PoE shares:
assert abs(sum(poe_dist) - 1.0) < 1e-12
assert poe_dist[1] == 0.0  # exact zero for hard-refused candidate
assert abs(poe_dist[0] - (0.504 / 0.584)) < 1e-12
assert abs(poe_dist[2] - (0.080 / 0.584)) < 1e-12



## 3. Optimal Verification Density: Why More Tests Can Hurt (§6.2)

Should we run every test and verify every intermediate step?

In Section 6.2, we model net task success $S(\alpha)$ as a function of **active tool/expert support density** $\alpha \in [0, 1]$ (the fraction of active tools or expert validators engaged in orchestration):

$$S(\alpha) = C(\alpha) \cdot (1 - D(\alpha))$$

- **$C(\alpha)$ (Coverage):** Increases with $\alpha$. More verification checks catch more bugs.
- **$D(\alpha)$ (Friction / Defect Accumulation):** Increases with $\alpha$. More verification checks cause context bloat, false refusals, latency timeouts, and evaluator gluing defects.

### The Interior First-Order Condition (FOC)
Taking the derivative with respect to $\alpha$ and setting $S'(\alpha) = 0$, any interior optimal verification density $\alpha^*$ satisfies:

$$C'(\alpha^*) \cdot (1 - D(\alpha^*)) = C(\alpha^*) \cdot D'(\alpha^*)$$

**Intuition:** At the optimum, the marginal coverage gained from adding one more test exactly equals the marginal loss from evaluator friction and false rejections!


In [ ]:
# Visualizing Verification Trade-Offs
alpha = np.linspace(0.01, 0.99, 200)

# Illustrative model: Coverage C(alpha) = alpha^0.5 (diminishing returns)
# Defect friction D(alpha) = 0.6 * alpha^1.5 (accelerating friction)
C = np.sqrt(alpha)
D = 0.6 * (alpha ** 1.5)
S = C * (1.0 - D)

# Find numerical optimum
opt_idx = np.argmax(S)
alpha_opt = alpha[opt_idx]
s_opt = S[opt_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8), dpi=120)

# Panel 1: The Activation-Density Optimum
ax1.plot(alpha, C, 'b--', label=r"Coverage $C(\alpha) = \sqrt{\alpha}$", linewidth=1.8)
ax1.plot(alpha, D, 'r:', label=r"Friction $D(\alpha) = 0.6 \alpha^{1.5}$", linewidth=1.8)
ax1.plot(alpha, S, 'g-', label=r"Net Success $S(\alpha) = C(1 - D)$", linewidth=2.5)
ax1.axvline(alpha_opt, color='black', linestyle='-.', alpha=0.7, label=f"Optimum $\\alpha^* = {alpha_opt:.2f}$")
ax1.scatter([alpha_opt], [s_opt], color='green', s=80, zorder=5)

ax1.set_xlabel(r"Active Support Density $\alpha$", fontsize=11, fontweight='bold')
ax1.set_ylabel("Probability / Rate", fontsize=11, fontweight='bold')
ax1.set_title("(a) Optimal Verification Density Trade-Off", fontsize=11, fontweight='bold')
ax1.legend(loc="lower left", fontsize=9)
ax1.grid(True, linestyle=':', alpha=0.6)

# Panel 2: PoE Refusal Accumulation as Experts (K) Grow
K_vals = np.arange(1, 15)
p_individual = 0.95  # 95% pass rate per individual expert
poe_pass = p_individual ** K_vals
poe_refusal = 1.0 - poe_pass

ax2.plot(K_vals, poe_pass, 'o-', color='#2b5c8f', label=r"PoE Acceptance $p^K$", linewidth=2)
ax2.plot(K_vals, poe_refusal, 's--', color='#d95f02', label=r"Complementary Refusal $1 - p^K$", linewidth=2)
ax2.set_xlabel("Number of Expert Evaluators (K)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Probability", fontsize=11, fontweight='bold')
ax2.set_title(r"(b) PoE Refusal Accumulation ($p=0.95$)", fontsize=11, fontweight='bold')
ax2.set_xticks(K_vals)
ax2.legend(fontsize=9.5)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 4. Mandatory Claim Hygiene: CAN-Show vs CANNOT-Show

### What this model CAN show:
- Shows that aggregate zero-residual ($R_{\mathrm{Eval}} = 0$) can arise from cancellation between positive and negative defects, rather than pointwise alignment.
- Shows that under an independent per-expert pass model, Product-of-Experts ensembles exponentially accumulate false refusals ($1 - p^K$) as the number of experts $K$ grows.
- Formulates the exact first-order condition $C'(1-D) = CD'$ governing interior optimal verification density.

### What this model CANNOT show:
- Does **not** claim that real LLM workflows follow exact square-root coverage curves (the activation model is a pedagogical formulation).
- Does **not** prove that quasi-concavity holds unconditionally without checking second-order derivative signs (see Section 6.2 single-crossing condition).


## 5. Student Exercises

### Exercise 1: Spotting False Cancellation
You run an EPK audit on a 100-trace agent benchmark and get $R_{\mathrm{Eval}} = 0.0000$.
Before declaring victory, what two checks must you perform?

<details>
<summary><b>Click to see the solution for Exercise 1</b></summary>

1. **Check Pointwise Defect Counts:** Verify that $\mathrm{NGD}_{\mathrm{count}} = 0$ and $\mathrm{PGD}_{\mathrm{count}} = 0$ (confirming $\mathrm{ZDR} = 100\%$), rather than $\mathrm{NGD}_{\mathrm{count}} = \mathrm{PGD}_{\mathrm{count}} > 0$ cancelling out.
2. **Run `DetectorAdequacyGuard.audit()`:** Verify that the evaluators are not degenerate (e.g. constant all-rejecting or tautological coupling).
</details>

### Exercise 2: PoE Calibration
If an agent pipeline chains $K = 8$ independent reviewers, each with a $98\%$ acceptance rate on valid code ($p = 0.98$):
What is the probability that a completely correct solution is wrongly rejected by the pipeline?

<details>
<summary><b>Click to see the solution for Exercise 2</b></summary>

- Pipeline acceptance rate: $P_{\mathrm{accept}} = p^K = (0.98)^8 \approx 0.8508 \quad (85.1\%)$.
- Probability of wrongful refusal:  
  $$P_{\mathrm{refuse}} = 1 - p^K = 1 - 0.8508 = 0.1492 \quad (14.9\%)$$
- Even with very accurate reviewers ($98\%$), nearly $15\%$ of good solutions are rejected purely through refusal accumulation!
</details>


---
## Anchors & Provenance

- **Paper Sections:** §4 (Evaluator Gluing Defect & Residual), §6.2 (Activation Density & Optimal Verification), §6.3 (Product-of-Experts vs Mixture-of-Experts), §11 (Proposal-Cost-Risk Kernels).
- **Claim Ledger IDs:** D4 (Cancellation not pointwise zero), F2 (Activation FOC), F4 (Product-of-Experts normalized distribution), F5 (PoE vs MoE positioning).
- **Code Artifacts:** `epk_core.ReliabilityModels`, `epk_core.EvaluatedPathKernel.compute_action_evaluator_residual`.
- **Lean 4 Mechanization:**
  - `PaperAE004.D4_cancellation_not_pointwise_zero` in `lean4/lggr_i_evaluated_path_kernels.lean`
  - `PaperAE004.P5_proposal_weighted_composition_under_G1_G2_G3_G4` in `lean4/lggr_i_evaluated_path_kernels.lean`
- **Next Step:** Open [`06_reviewer_reference_verification.ipynb`](06_reviewer_reference_verification.ipynb) for the headless peer reviewer verification harness.
